# Hypothesis-driven document analysis

Select the project kernel and use **Run All**. This notebook reads the frozen
20,000-document development cohort (19,998 with nonzero features) and the same
5,000-document validation panel. The 54 fitted comparisons and dated failure
ledger remain available as an earlier corpus-structure experiment. The current
default run produces hypothesis diagnostics and a training review roster.

The macro hypothesis concerns social learning and cultural transmission. Its
evidence requires separate context, attribution and temporal ordering tests.
[Accepted research questions](../docs/research/2026-11-13-corpus-release/RESEARCH_QUESTIONS.md)
· [Earlier comparison and its limitations](../docs/research/2026-11-22-document-comparison/EXPERIMENT_LOG.md)

## Hypotheses and evidence boundaries

| Research aim | Evidence inspected in this pilot | Decision boundary |
| --- | --- | --- |
| RQ1 · Heat-related affect over time | Explicit heat or warming language near Fear, worry, or bodily discomfort cues. Report 80- and 160-character gaps in the same paragraph and fixed host and format composition across periods. | Examine changes in both directions. Cue hits are text diagnostics. Independent human decisions establish expressed affect and its target. |
| RQ1 · Cultural-learning evidence | Separate heat-linked memory and transmission cues from current reactions. Record quotation and negation markers for contextual review. | Recurrence, independent source histories, textual attribution and chronology are required for transmission evidence. |
| RQ2 · Measurement validity | Use policy and sports controls, ambiguous cases and a cue-absent sample in a training-only review queue. Compare a transparent rule with a conditional classifier after human adjudication. | Keep validation independent of model and cue selection. Expressed Fear, worry and discomfort are different outcomes. |
| RQ3 · Event response | Check the provisional registry and presence of independently documented windows. | Publication year supplies inventory only. Event exposure and day-level effects remain pending. |

The cue dictionary, distance thresholds, periods and denominator choices are
fixed before the new diagnostic run. Source and format adjustment is conditional
on common support. Capture process, version and event adjustments need their
own bound covariates. The test split remains reserved.

The directional expectation for the cultural-learning hypothesis concerns
independently verified recurrence and circulation across sources. A rise in
total Fear expression is not a required prediction. A temporal pattern
confined to one source, lost after composition adjustment, or explained by
quoted or unrelated expressions weakens a shared-discourse interpretation.

In [ ]:
print("Notebook 16 kernel is running. Verifying the fixed document panel.", flush=True)

import hashlib
import json
import tempfile
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/15_unsupervised_research_plan_and_local_pilot.ipynb").is_file()
)
NOTEBOOK = ROOT / "notebooks/16_document_unsupervised_comparison.ipynb"
# The researcher enables this cell branch after independent adjudication.
RUN_LABELLED_PILOT = False
SEED = 1729
DRAWS = 300
CUE_GAPS = (80, 160)
PERIODS = ("2003–2010", "2011–2017", "2018–2025")
COMMON_STRATUM_MINIMUM = 5
REVIEW_PER_STRATUM = 8

def sha256_file(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def digest(value):
    data = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(data).hexdigest()

def verify_manifest(folder, expected=None):
    manifest_path = folder / "MANIFEST.json"
    if expected is not None and sha256_file(manifest_path) != expected:
        raise ValueError(f"The frozen manifest changed: {folder.name}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for item in manifest["files"]:
        if sha256_file(folder / item["path"]) != item["sha256"]:
            raise ValueError(f"A recorded file failed verification: {item['path']}")
    return manifest

previous_runs = ROOT / ".cache/document-model-comparison"
input_root = previous_runs / "ee0a82a0e8ce8f3b90d0/document_input"
input_manifest_sha = "0c702b8bfa694f10aa258d2a54266daf237361774e1cc567f3930103aa06e7c2"
source_report = previous_runs / "faba1e820b243b531d3f/report-734728858f6801"
source_report_sha = "a67c631c9947033e221107db66156e96d000c21b2ca83bbd431b14a167855b30"
if not (input_root / "MANIFEST.json").is_file():
    raise FileNotFoundError(
        "Restore the sealed document_input snapshot from the original comparison. "
        "See the Notebook 16 section of PREPROCESSING_START_HERE.md."
    )
input_manifest = verify_manifest(input_root, input_manifest_sha)
if (source_report / "MANIFEST.json").is_file():
    print("The earlier comparison is available locally. Its results are recorded in the experiment log.")

hold_root = ROOT / "work_packages/formal_corpus_release_20261117/structure-v2-source-use-v1"
hold_manifest = json.loads((hold_root / "MANIFEST.json").read_text(encoding="utf-8"))
hold_parts = hold_manifest["table_parts_sha256"]["source_use_body_holds"]
for name, checksum in hold_parts.items():
    if sha256_file(hold_root / name) != checksum:
        raise ValueError("The source-use hold inventory failed verification.")
held_rows = set(pq.read_table(
    [hold_root / name for name in hold_parts], columns=["release_row"]
)["release_row"].to_pylist())
if len(held_rows) != 732:
    raise ValueError("The source-use hold roster changed.")

metadata_columns = [
    "release_row", "clean_sha256", "release_group_id", "source_host", "format", "publication_year",
]
inputs = {}
split_paths = {}
for split, size in (("train", 19_998), ("validation", 5_000)):
    paths = [
        input_root / item["path"] for item in input_manifest["files"]
        if item["path"].startswith(f"rows/ml_split={split}/") and item["path"].endswith(".parquet")
    ]
    split_paths[split] = paths
    panel = pq.read_table(paths, columns=[*metadata_columns, "clean_text"]).sort_by("release_row").to_pandas()
    if (
        len(panel) != size or not panel.release_row.is_unique or panel.release_group_id.isna().any()
        or panel.clean_text.isna().any() or set(panel.release_row) & held_rows
    ):
        raise ValueError(f"The {split} panel did not reconcile with the admitted cohort.")
    inputs[split] = panel
if set(inputs["train"].release_group_id) & set(inputs["validation"].release_group_id):
    raise ValueError("Train and validation share a release group.")
print("Verified the 19,998 training and 5,000 validation documents.")

## Observable cue pairs and counterexamples

Read the unchanged document text and preserve half-open code-point offsets.
A candidate is the closest pair in a paragraph. A heat anchor can refer to
an unrelated object, a concern cue can refer to politics, and a quotation can
describe someone else's experience. Preserve these cases in the review pool.
Policy and sports controls use paragraphs without a heat anchor. The bodily
cue list checks for discomfort language, including cases that may have a
medical explanation. Cue absence remains eligible for human review.

In [ ]:
import re
from bisect import bisect_left

# Development cues chosen for auditability. They do not create labels.
cue_patterns = {
    "heat": r"heat[\s-]?waves?|heat|hot\s+weather|hotter|extreme\s+temperatures?|rising\s+temperatures?|warming",
    "fear": r"fear(?:s|ed|ful)?|afraid|scared|terrified|terror|panic(?:ked|king)?",
    "worry": r"worr(?:y|ies|ied|ying)|anxious|anxiety|concerned|dread(?:ing|ed)?",
    "somatic": r"sweat(?:ing|y)?|dehydrat(?:ed|ion)|heatstroke|can(?:not|['’]t)\s+cool\s+down|couldn['’]t\s+sleep",
    "memory": r"remember\s+when|i\s+remember|we\s+remember|growing\s+up|back\s+then|used\s+to\s+be",
    "transmission": r"taught\s+us|we\s+were\s+taught|my\s+parents\s+told\s+me|pass(?:ed|ing)?\s+down|"
                    r"our\s+grandparents|for\s+our\s+(?:kids|children)",
    "policy": r"carbon\s+tax|carbon\s+pricing|emissions\s+trading|renewable\s+energy|climate\s+policy",
    "sport": r"afl|grand\s+final|footy|match\s+day|the\s+swans|the\s+referee",
    "negative_context": r"hate|furious|angry|worr(?:y|ied)|fear|afraid|disgusted",
}
cues = {key: re.compile(rf"\b(?:{pattern})\b", re.IGNORECASE) for key, pattern in cue_patterns.items()}
paragraph_break = re.compile(r"\r?\n[ \t]*\r?\n(?:[ \t]*\r?\n)*")
negation = re.compile(r"\b(?:not|no|never|don['’]t|isn['’]t|without)\b", re.IGNORECASE)
quotes = re.compile(r'["“”]')
heat_channels = ("fear", "worry", "somatic", "memory", "transmission")
evidence_channels = tuple(f"heat_{name}" for name in heat_channels) + (
    "policy_negative_control", "sports_negative_control",
)

def closest_pair(left, right):
    starts = [match.start() for match in right]
    best = None
    for first in left:
        index = bisect_left(starts, first.start())
        for nearby in right[max(0, index - 1):index + 1]:
            gap = max(first.start() - nearby.end(), nearby.start() - first.end(), 0)
            pair = (gap, first.start(), first.end(), nearby.start(), nearby.end())
            if best is None or pair < best:
                best = pair
    return best

def find_evidence(text):
    found = {}
    heat_opportunity = False
    start = 0
    ends = [(match.start(), match.end()) for match in paragraph_break.finditer(text)]
    for end, following in [*ends, (len(text), len(text))]:
        paragraph = text[start:end]
        heat = list(cues["heat"].finditer(paragraph))
        heat_opportunity |= bool(heat)
        if heat:
            pairs = (
                (f"heat_{name}", heat, list(cues[name].finditer(paragraph)))
                for name in heat_channels
            )
        else:
            negative = list(cues["negative_context"].finditer(paragraph))
            pairs = (
                (channel, list(cues[anchor].finditer(paragraph)), negative)
                for channel, anchor in (("policy_negative_control", "policy"), ("sports_negative_control", "sport"))
            )
        for channel, left, right in pairs:
            best = closest_pair(left, right)
            if best is None or best[0] > CUE_GAPS[-1]:
                continue
            gap, a, b, c, d = best
            absolute = (gap, start + a, start + b, start + c, start + d)
            if channel not in found or absolute < found[channel]:
                found[channel] = absolute
        start = following
    return heat_opportunity, found

# Small boundary controls before any real documents are scanned.
assert "heat_fear" in find_evidence("I am afraid of this heatwave.")[1]
assert "heat_fear" not in find_evidence("The heatwave ended.\n\nI am afraid of losing the grand final.")[1]
assert "sports_negative_control" in find_evidence("I hate losing the grand final.")[1]
assert "heat_fear" in find_evidence('She said "I am not afraid of heat".')[1]

cues_by_split = {}
evidence_by_split = {}
for split, panel in inputs.items():
    rows, evidence = [], []
    for row, text in zip(panel.release_row, panel.clean_text, strict=True):
        opportunity, matches = find_evidence(text)
        item = {"release_row": row, "heat_cue_document": opportunity}
        for channel in evidence_channels:
            pair = matches.get(channel)
            for radius in CUE_GAPS:
                item[f"{channel}_{radius}"] = pair is not None and pair[0] <= radius
            if pair is None:
                continue
            gap, a, b, c, d = pair
            assert 0 <= min(a, c) < max(b, d) <= len(text)
            context = text[max(0, min(a, c) - 48):max(b, d) + 48]
            evidence.append({
                "release_row": row, "cue_pair": channel, "gap_codepoints": gap,
                "anchor_start": a, "anchor_end": b, "cue_start": c, "cue_end": d,
                "nearby_negation_marker": bool(negation.search(context)),
                "nearby_quotation_marker": bool(quotes.search(context)),
            })
        rows.append(item)
    cues_by_split[split] = pd.DataFrame(rows).merge(
        panel[metadata_columns], on="release_row", validate="1:1"
    )
    evidence_by_split[split] = pd.DataFrame(evidence, columns=[
        "release_row", "cue_pair", "gap_codepoints", "anchor_start", "anchor_end", "cue_start", "cue_end",
        "nearby_negation_marker", "nearby_quotation_marker",
    ])
    print(f"{split}: completed {len(rows):,} document diagnostics.")

## RQ1 · What changes across publication periods?

Report paired-cue rates per document and, for the heat outcomes, per
heat-cue document. Compare the observed validation mix with direct
standardisation to a pooled mix of shared source and format strata.
A common stratum contributes at least five denominator documents per
period. Reweight the same release groups in the bootstrap across periods.
The intervals describe the held-out panel conditional on these cue rules.
For endpoint comparisons with zero hits in either supported period, retain the observed difference and mark the resampling interval unavailable. The negative-control and distance contrasts remain visible in the report.

In [ ]:
from itertools import product

validation = cues_by_split["validation"].copy()
validation["period"] = pd.cut(
    validation.publication_year, [2002, 2010, 2017, 2025], labels=PERIODS
)
dated = validation.loc[validation.period.notna()].copy()
dated["period"] = dated.period.astype(str)
stratum_columns = ["source_host", "format"]
validation_groups = pd.Index(dated.release_group_id.unique())
group_multipliers = np.random.default_rng(SEED).poisson(1, (DRAWS, len(validation_groups)))
models = [(channel, radius, "all_documents") for channel, radius in product(evidence_channels, CUE_GAPS)]
models += [
    (f"heat_{name}", radius, "heat_cue_documents")
    for name, radius in product(("fear", "worry", "transmission"), CUE_GAPS)
]

def bootstrap_share(rows, outcome):
    counts = group_multipliers[:, validation_groups.get_indexer(rows.release_group_id)]
    denominator = counts.sum(axis=1)
    return np.divide(
        counts @ rows[outcome].to_numpy(dtype=float), denominator,
        out=np.full(DRAWS, np.nan), where=denominator > 0,
    )

def summarise_rates(channel, radius, denominator):
    outcome = f"{channel}_{radius}"
    selected = dated.loc[dated.heat_cue_document] if denominator == "heat_cue_documents" else dated
    known = selected.dropna(subset=stratum_columns)
    strata = known.groupby([*stratum_columns, "period"]).size().unstack(fill_value=0)
    strata = strata.reindex(columns=PERIODS, fill_value=0)
    shared = strata.loc[(strata >= COMMON_STRATUM_MINIMUM).all(axis=1)]
    weights = shared.sum(axis=1)
    weights = (weights / weights.sum()).rename("weight").reset_index()
    support = known.merge(weights, on=stratum_columns, how="inner")
    result, draws = [], {}
    for period in PERIODS:
        actual = selected.loc[selected.period == period]
        matched = support.loc[support.period == period]
        rate = np.nan
        if not matched.empty:
            rate = 0.0
            simulated = np.zeros(DRAWS)
            for _, stratum in matched.groupby(stratum_columns):
                weight = float(stratum.weight.iloc[0])
                rate += weight * float(stratum[outcome].mean())
                simulated += weight * bootstrap_share(stratum, outcome)
            draws[period] = simulated
        result.append({
            "cue_pair": channel, "gap_codepoints": radius, "denominator": denominator, "period": period,
            "documents": len(actual), "cue_pair_documents": int(actual[outcome].sum()),
            "observed_rate": float(actual[outcome].mean()),
            "common_support_documents": len(matched),
            "common_support_cue_documents": int(matched[outcome].sum()),
            "common_source_format_strata": len(shared),
            "common_support_coverage": len(matched) / len(actual) if len(actual) else np.nan,
            "standardised_rate": rate,
        })
    contrast = None
    if PERIODS[0] in draws and PERIODS[-1] in draws:
        delta = draws[PERIODS[-1]] - draws[PERIODS[0]]
        endpoint_hits = [result[i]["common_support_cue_documents"] for i in (0, -1)]
        finite = delta[np.isfinite(delta)]
        supported_interval = min(endpoint_hits) > 0 and len(finite) > 0
        low, high = np.quantile(finite, [0.025, 0.975]) if supported_interval else [np.nan, np.nan]
        contrast = {
            "cue_pair": channel, "gap_codepoints": radius, "denominator": denominator,
            "late_minus_early": result[-1]["standardised_rate"] - result[0]["standardised_rate"],
            "early_support_cue_documents": endpoint_hits[0],
            "late_support_cue_documents": endpoint_hits[1],
            "bootstrap_95_low": low, "bootstrap_95_high": high,
            "valid_bootstrap_draws": len(finite) if supported_interval else 0,
            "interval_status": "available" if supported_interval else "insufficient_endpoint_events_or_draws",
        }
    return result, contrast

period_rows, differences = [], []
for model in models:
    estimates, difference = summarise_rates(*model)
    period_rows.extend(estimates)
    if difference is not None:
        differences.append(difference)
rq1_period = pd.DataFrame(period_rows)
rq1_differences = pd.DataFrame(differences, columns=[
    "cue_pair", "gap_codepoints", "denominator", "late_minus_early",
    "early_support_cue_documents", "late_support_cue_documents",
    "bootstrap_95_low", "bootstrap_95_high", "valid_bootstrap_draws", "interval_status",
])
annual = dated.groupby("publication_year", as_index=False).agg(
    **{f"{c}_{r}": (f"{c}_{r}", "sum") for c, r in product(evidence_channels, CUE_GAPS)},
    documents=("release_row", "size"), heat_cue_documents=("heat_cue_document", "sum"),
)
composition = dated.groupby(["period", *stratum_columns], dropna=False).size().rename("documents").reset_index()
show = rq1_period.loc[
    rq1_period.cue_pair.isin(["heat_fear", "heat_worry", "heat_transmission", "policy_negative_control"])
    & rq1_period.gap_codepoints.eq(80) & rq1_period.denominator.eq("all_documents")
]
display(show.reset_index(drop=True))
print(f"Validation documents outside the publication windows: {len(validation) - len(dated):,}")

## Human construct review and codebook draft

Review one or more original passages with context, using the verified document
hash and offsets. Record the target first, then the affect and whose expression
it represents. Separate worry about a prospective harm, Fear toward a threat,
bodily discomfort, and other negative affect. Mark ambiguous, hypothetical,
negated and reported or quoted cases explicitly. Record an evidence span and
a rationale. Do not use the cue pair or a model prediction as a reviewer label.

The training queue prioritises contrasting evidence cells across periods and
hosts. The cue-absent stratum can reveal missing vocabulary and implicit language.
This is a purposive codebook pilot. Human review should revise the rubric,
track disagreements and adjudicate them before using labels for learning.

In [ ]:
training = cues_by_split["train"].copy()
training["period"] = pd.cut(
    training.publication_year, [2002, 2010, 2017, 2025], labels=PERIODS
).astype("string").fillna("Unknown")
queue_bands = {
    "direct_fear_cue": training.heat_fear_80,
    "worry_cue": training.heat_worry_80,
    "bodily_cue": training.heat_somatic_160,
    "memory_or_transmission_cue": training.heat_memory_160 | training.heat_transmission_160,
    "policy_control": training.policy_negative_control_160,
    "sports_control": training.sports_negative_control_160,
    "heat_without_nearby_affect_cue": training.heat_cue_document & ~(
        training.heat_fear_160 | training.heat_worry_160 | training.heat_somatic_160
    ),
    "no_registered_cue": ~training[["heat_cue_document", *[f"{c}_160" for c in evidence_channels]]].any(axis=1),
}
training["draw_key"] = training.clean_sha256.map(lambda sha: digest([SEED, sha]))
used_groups, queue, queue_balance = set(), [], []
for band, mask in queue_bands.items():
    for period, pool in training.loc[mask].groupby("period", sort=True):
        candidates = pool.loc[~pool.release_group_id.isin(used_groups)].sort_values("draw_key")
        chosen = (
            candidates.drop_duplicates("source_host").drop_duplicates("release_group_id")
            .head(REVIEW_PER_STRATUM)
        )
        used_groups.update(chosen.release_group_id)
        queue.append(chosen[[*metadata_columns, "period"]].assign(review_band=band))
        queue_balance.append({"review_band": band, "period": period,
                              "candidates": len(pool), "selected": len(chosen)})
review_roster = pd.concat(queue, ignore_index=True)
if (
    not review_roster.release_group_id.is_unique
    or set(review_roster.release_group_id) & set(validation.release_group_id)
):
    raise ValueError("The training-only review roster does not preserve release groups.")
review_offsets = evidence_by_split["train"].merge(
    review_roster[["release_row", "clean_sha256", "review_band"]], on="release_row", how="inner"
)
review_form = review_roster[["release_row", "clean_sha256", "review_band"]].assign(
    ml_split="train", target="", affect="", attribution="", assertion="",
    evidence_start="", evidence_end="", adjudication_status="pending", rationale="",
)
display(pd.DataFrame(queue_balance).groupby("review_band").selected.sum().reset_index())

### Inspect one training review document

Set **REVIEW_DOCUMENT** to a row from the training review roster and run the next cell. The display is local and includes the full verified text, without a cue or model label. Clear that cell output before sharing a notebook.

In [ ]:
REVIEW_DOCUMENT = None
if REVIEW_DOCUMENT is None:
    print("Set REVIEW_DOCUMENT to a training review release row to display its evidence.")
else:
    selected = review_roster.loc[review_roster.release_row.eq(REVIEW_DOCUMENT)]
    if len(selected) != 1:
        raise ValueError("Select a release row from the training review roster.")
    document = inputs["train"].loc[inputs["train"].release_row.eq(REVIEW_DOCUMENT)].iloc[0]
    if hashlib.sha256(document.clean_text.encode("utf-8")).hexdigest() != document.clean_sha256:
        raise ValueError("The displayed text failed its recorded hash.")
    print(document.clean_text)

## Conditional learning once an independent label panel exists

The optional transparent baseline uses the existing train-only TF–IDF
vocabulary, a fixed regularised logistic model, and the predefined narrow
cue rule as a comparator. The response is affirmed, attributable heat-related
Fear or worry. The stored adjudication keeps Fear, worry, physical discomfort
and other affect distinct. Uncertain documents remain unscored. Report
held-out precision, recall, F1 and, for the logistic baseline, average
precision and Brier loss. An exploratory held-out result does not certify
calibration or licence full-corpus projection.

After independent human adjudication, supply **adjudicated.csv** and
**MANIFEST.json** under `.cache/hypothesis-document-pilot/annotations/` and
enable `RUN_LABELLED_PILOT` in the setup cell. The manifest includes the
frozen document input manifest SHA-256, the CSV hash, a codebook version and
the validation selection protocol, frozen before model predictions. Review
queues from this notebook belong to training only.

In [ ]:
annotation_root = ROOT / ".cache/hypothesis-document-pilot/annotations"
label_columns = (
    "release_row", "clean_sha256", "ml_split", "target", "affect", "attribution",
    "assertion", "evidence_start", "evidence_end", "adjudication_status",
)
label_codebook = {
    "target": ["heat_or_warming", "other_climate", "unrelated", "uncertain"],
    "affect": ["fear", "worry", "discomfort", "other_negative", "none", "uncertain"],
    "attribution": ["author", "reported_or_quoted", "none", "uncertain"],
    "assertion": ["affirmed", "negated", "hypothetical", "none", "uncertain"],
}
label_contract = {
    "required_columns": label_columns, "codebook_values": label_codebook,
    "adjudication_status": "adjudicated", "manifest_required": [
        "document_input_manifest_sha256", "labels_sha256", "codebook_version",
        "validation_selection_protocol",
    ],
}
evaluation_metrics = None
assessment_predictions = None
learning_status = "awaiting_independent_adjudication"
annotation_sha = None

def verified_adjudications(directory):
    manifest_path = directory / "MANIFEST.json"
    csv_path = directory / "adjudicated.csv"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if (
        manifest["document_input_manifest_sha256"] != input_manifest_sha
        or manifest["labels_sha256"] != sha256_file(csv_path)
        or not str(manifest["codebook_version"]).strip()
        or not str(manifest["validation_selection_protocol"]).strip()
    ):
        raise ValueError("The human labels need a bound codebook and validation selection record.")
    reviews = pd.read_csv(csv_path, keep_default_na=False)
    if (
        not set(label_columns) <= set(reviews.columns) or not reviews.release_row.is_unique
        or set(reviews.ml_split) != {"train", "validation"}
        or not reviews.adjudication_status.eq("adjudicated").all()
    ):
        raise ValueError("The adjudicated file must contain one labelled row per document across both partitions.")
    for field, levels in label_codebook.items():
        if not reviews[field].isin(levels).all():
            raise ValueError(f"Unknown adjudication category for {field}.")
    result = {}
    for split, panel in inputs.items():
        rows = reviews.loc[reviews.ml_split.eq(split)].copy()
        positions = pd.Index(panel.release_row).get_indexer(rows.release_row)
        if (positions < 0).any():
            raise ValueError(f"The {split} adjudicated labels cross the frozen input boundary.")
        bodies = panel.iloc[positions]
        if not np.array_equal(rows.clean_sha256.to_numpy(), bodies.clean_sha256.to_numpy()):
            raise ValueError(f"The {split} adjudicated labels refer to different text.")
        uncertain = rows[list(label_codebook)].eq("uncertain").any(axis=1)
        target = (
            rows.target.eq("heat_or_warming") & rows.affect.isin(["fear", "worry"])
            & rows.attribution.eq("author") & rows.assertion.eq("affirmed")
        )
        lengths = bodies.clean_text.str.len().to_numpy()[target.to_numpy()]
        starts = pd.to_numeric(rows.loc[target, "evidence_start"], errors="raise").to_numpy(dtype=float)
        ends = pd.to_numeric(rows.loc[target, "evidence_end"], errors="raise").to_numpy(dtype=float)
        if (
            not np.isfinite(starts).all() or not np.isfinite(ends).all()
            or (starts % 1 != 0).any() or (ends % 1 != 0).any()
            or (starts < 0).any() or (starts >= ends).any() or (ends > lengths).any()
        ):
            raise ValueError("Positive adjudications require valid offsets in the bound document text.")
        result[split] = (positions[~uncertain], target.loc[~uncertain].to_numpy(), int(uncertain.sum()))
    return result, sha256_file(manifest_path)

def load_tfidf_rows(split, positions):
    from scipy.sparse import csr_matrix

    source = pq.read_table(split_paths[split], columns=["release_row", "tfidf"]).sort_by("release_row")
    if not np.array_equal(source["release_row"].to_numpy(), inputs[split].release_row.to_numpy()):
        raise ValueError("The TF–IDF rows do not match the verified text panel.")
    vectors = source.take(positions)["tfidf"].combine_chunks()
    index = vectors.field("indices")
    value = vectors.field("values")
    if not np.all(vectors.field("type").to_numpy() == 0) or not np.array_equal(
        index.offsets.to_numpy(), value.offsets.to_numpy()
    ):
        raise ValueError("The pilot expects valid sparse vector rows.")
    return csr_matrix(
        (value.flatten().to_numpy().astype(np.float32), index.flatten().to_numpy(), index.offsets.to_numpy()),
        shape=(len(positions), 24_000),
    )

if RUN_LABELLED_PILOT:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_fscore_support

    adjudicated, annotation_sha = verified_adjudications(annotation_root)
    train_positions, train_labels, train_uncertain = adjudicated["train"]
    val_positions, val_labels, val_uncertain = adjudicated["validation"]
    if len(np.unique(train_labels)) != 2 or len(np.unique(val_labels)) != 2:
        raise ValueError("Both outcomes must occur in both adjudicated splits for this comparison.")
    classifier = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    classifier.fit(load_tfidf_rows("train", train_positions), train_labels)
    scores = classifier.predict_proba(load_tfidf_rows("validation", val_positions))[:, 1]
    rule_panel = cues_by_split["validation"].iloc[val_positions]
    rule_prediction = (rule_panel.heat_fear_80 | rule_panel.heat_worry_80).to_numpy()
    estimates = []
    for name, prediction in [("paired_cue_rule", rule_prediction), ("tfidf_logistic", scores >= 0.5)]:
        precision, recall, f1, _ = precision_recall_fscore_support(
            val_labels, prediction, average="binary", zero_division=0,
        )
        estimates.append({
            "model": name, "validation_documents": len(val_labels), "positive_documents": int(val_labels.sum()),
            "precision": precision, "recall": recall, "f1": f1,
            "average_precision": average_precision_score(val_labels, scores) if name == "tfidf_logistic" else np.nan,
            "brier_loss": brier_score_loss(val_labels, scores) if name == "tfidf_logistic" else np.nan,
            "excluded_uncertain_train": train_uncertain, "excluded_uncertain_validation": val_uncertain,
        })
    evaluation_metrics = pd.DataFrame(estimates)
    assessment_predictions = inputs["validation"].iloc[val_positions][metadata_columns].assign(
        adjudicated_target=val_labels, cue_prediction=rule_prediction, logistic_score=scores,
    )
    learning_status = "independent_label_pilot_only"
    display(evaluation_metrics)
else:
    print("Learning stage awaits independently adjudicated labels and an operator-started pilot.")

## Event readiness and run record

Event contrasts require independent dates, geographic scope, a fixed
pre/during/post policy and verified document date precision. Cultural
transmission requires separate contextual evidence. Keep the cue results,
review pool and any labelled development comparison bound to one run.

In [ ]:
event_path = ROOT / "docs/research/2026-11-13-prechunk-pilot/event_registry_draft.json"
events = json.loads(event_path.read_text(encoding="utf-8"))
event_readiness = pd.DataFrame([
    {
        "event_id": event["event_id"], "provisional_label": event["user_label"],
        "official_evidence_present": bool(event["official_evidence"]),
        "anchor_date_present": bool(event["event_anchor_date"]),
        "analysis_window_present": bool(event["analysis_window_start"] and event["analysis_window_end"]),
    }
    for event in events["events"]
])
covariate_readiness = {
    "publication_year": "available", "source_and_format": "common_support_analysis",
    "capture_and_version": "awaiting_bound_covariates", "event_exposure": "awaiting_frozen_windows",
    "affect_measurement": learning_status, "event_comparisons_run": 0,
}
display(event_readiness)

## Export the hypothesis pilot

The report stores evidence keys and offsets, cue rates, group-bootstrap
contrasts and a blank review form. It carries the exact input and protocol
fingerprints. The figure shows descriptive rates for the same validation
documents under both distance definitions and within the supported mix.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image

run_identity = {
    "notebook_sources_sha256": digest([
        (cell.id, cell.source) for cell in nbformat.read(NOTEBOOK, as_version=4).cells
    ]),
    "document_input_manifest_sha256": input_manifest_sha,
    "source_use_manifest_sha256": sha256_file(hold_root / "MANIFEST.json"),
    "initial_comparison_manifest_sha256": source_report_sha,
    "event_registry_sha256": sha256_file(event_path),
    "adjudication_manifest_sha256": annotation_sha,
    "cue_patterns": cue_patterns, "gap_codepoints": CUE_GAPS, "periods": PERIODS,
    "common_stratum_minimum": COMMON_STRATUM_MINIMUM,
    "review_per_band_and_period": REVIEW_PER_STRATUM, "bootstrap_draws": DRAWS, "seed": SEED,
}
output_base = ROOT / ".cache/hypothesis-document-pilot"
output_base.mkdir(mode=0o700, parents=True, exist_ok=True)
run_directory = output_base / digest(run_identity)[:20]
if not run_directory.exists():
    with tempfile.TemporaryDirectory(prefix=".hypothesis-report-", dir=output_base) as directory:
        staged = Path(directory) / "report"
        staged.mkdir(mode=0o700)
        frames = {
            "rq1_period_cue_rates": rq1_period, "rq1_period_contrasts": rq1_differences,
            "rq1_annual_cue_counts": annual, "rq1_source_format_composition": composition,
            "training_review_roster": review_roster, "training_review_evidence_offsets": review_offsets,
            "training_review_selection_counts": pd.DataFrame(queue_balance),
            "training_blank_review_form": review_form, "event_readiness": event_readiness,
        }
        for name, frame in frames.items():
            frame.to_csv(staged / f"{name}.csv", index=False)
        for split, evidence in evidence_by_split.items():
            evidence.to_parquet(staged / f"{split}_cue_offsets.parquet", index=False)
        if evaluation_metrics is not None:
            evaluation_metrics.to_csv(staged / "labelled_pilot_metrics.csv", index=False)
            assessment_predictions.to_parquet(staged / "labelled_pilot_validation_predictions.parquet", index=False)
        (staged / "annotation_contract.json").write_text(
            json.dumps(label_contract, indent=2, ensure_ascii=False) + "\n", encoding="utf-8",
        )
        (staged / "research_status.json").write_text(
            json.dumps(covariate_readiness, indent=2) + "\n", encoding="utf-8",
        )
        figure, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True, constrained_layout=True)
        for axis, measure, title in zip(
            axes, ("observed_rate", "standardised_rate"), ("Observed validation mix", "Shared source and format mix"),
            strict=True,
        ):
            for channel in ("heat_fear", "heat_worry", "heat_transmission", "policy_negative_control"):
                part = rq1_period.loc[
                    rq1_period.cue_pair.eq(channel) & rq1_period.denominator.eq("all_documents")
                ]
                for radius, style in zip(CUE_GAPS, ("-", "--"), strict=True):
                    values = part.loc[part.gap_codepoints.eq(radius)].set_index("period").reindex(PERIODS)
                    axis.plot(PERIODS, values[measure] * 100, linestyle=style, marker="o",
                              label=f"{channel.replace('_', ' ')} · {radius} chars")
            axis.set_title(title)
            axis.set_xlabel("Publication period")
            axis.grid(axis="y", alpha=0.2)
        axes[0].set_ylabel("Documents with a cue pair (%)")
        axes[1].legend(fontsize=7)
        figure.savefig(staged / "rq1_cue_rates.png", dpi=180, facecolor="white")
        figure.savefig(staged / "rq1_cue_rates.pdf", facecolor="white")
        plt.close(figure)
        manifest = {
            "identity": run_identity,
            "files": [{"path": file.name, "sha256": sha256_file(file)} for file in sorted(staged.iterdir())],
        }
        (staged / "MANIFEST.json").write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8",
        )
        if run_directory.exists():
            raise FileExistsError("Another run already sealed this report.")
        staged.rename(run_directory)
verified = verify_manifest(run_directory)
if digest(verified["identity"]) != digest(run_identity):
    raise ValueError("The saved report belongs to a different pilot.")
display(Image(filename=str(run_directory / "rq1_cue_rates.png"), width=920))
print(f"Report verified at {run_directory.relative_to(ROOT)}")